# midi2Frets — Guitar Tab Transformer (Colab Trainer)

Train the string-prediction model on the full Guitar Pro corpus using a free Colab GPU (T4, 16 GB — far faster than a local 1650 Ti).

**Before you start:** upload the project folder (the one containing `run.py`, `src/`, `data/`) to your Google Drive, e.g. `MyDrive/midi2Frets`. You can upload just `src/` + `data/raw` + `data/processed/gp_json` if you already preprocessed locally; or `src/` + the raw `.gp*` files and let this notebook preprocess on Colab's CPUs.

Then: **Runtime → Change runtime type → GPU**, and run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Install the one missing dependency (PyGuitarPro)
Torch + CUDA are already installed on Colab.

In [ ]:
!pip -q install PyGuitarPro pyyaml

## 3. Mount Google Drive and enter the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# >>> EDIT THIS to wherever you uploaded the project on your Drive <<<
PROJECT = '/content/drive/MyDrive/midi2Frets'
os.chdir(PROJECT)
print('cwd:', os.getcwd())
print('contents:', sorted(os.listdir())[:20])

## 4. Preprocess raw .gp* → JSON (REQUIRED this time: `--fresh`)

The preprocessing was fixed to write **one JSON per guitar track** instead of merging all guitars of a song into one corrupted stream (~80% of the corpus was affected). Old JSONs must be wiped and regenerated once — that is what `--fresh` does. After this one-time rebuild, plain `python run.py preprocess` is resumable again.

This needs the raw `.gp*` files on Drive (e.g. `data/ScoreSetDataSet/`). It takes a while on Colab CPUs — but only once.

In [ ]:
# ONE-TIME after the track-split fix: wipe old merged JSONs + chunk index and rebuild
!python run.py preprocess --fresh --workers 2

## 5. Quick sanity check (optional, ~1 min)
Overfit a single song; expect string accuracy > 95%. Confirms the whole pipeline works before a long run.

In [ ]:
!python run.py overfit --device cuda

## 6. Train on the full corpus (streaming, song-level split)
Streaming = no RAM cap, so all songs are used. The first run builds a chunk index (cached to Drive; instant next time). Live progress prints below; it also writes `checkpoints/logs/training.log` and `metrics.jsonl` on your Drive.

T4 handles a big batch. Auto-stops on convergence (patience) so you don't over-train.

In [ ]:
# NOTE: after the --fresh preprocess, save to a NEW checkpoint name so the old
# model stays available for comparison until the new one evaluates better.
!python run.py train \
    --device cuda \
    --batch 128 \
    --num-workers 2 \
    --epochs 30 \
    --patience 6 \
    --eval-batches 100 \
    --save checkpoints/model_gp_v2.pt

### Resume an interrupted run
Colab can disconnect. Just re-run with `--resume` — the chunk index is cached and the optimizer/epoch state is restored.

In [ ]:
!python run.py train --device cuda --batch 128 --num-workers 2 --epochs 30 \
    --save checkpoints/model_gp_v2.pt \
    --resume checkpoints/model_gp_v2.pt.resume

## 7. Evaluate + render a tab (model vs DP baseline vs human)

In [ ]:
!python src/evaluate.py --data data/raw/file.json \
    --checkpoint checkpoints/model_gp_v2.pt \
    --render --out data/processed/eval_colab.txt
print(open('data/processed/eval_colab.txt', encoding='utf-8').read())

## 8. Plot the training curves from metrics.jsonl

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('checkpoints/logs/metrics.jsonl', encoding='utf-8')]
val = [r for r in rows if r.get('split') == 'val']
tr  = [r for r in rows if r.get('split') == 'train']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r['step'] for r in tr], [r['loss'] for r in tr], label='train', alpha=.5)
ax[0].plot([r['step'] for r in val], [r['loss'] for r in val], label='val', marker='o')
ax[0].set_title('loss'); ax[0].set_xlabel('step'); ax[0].legend()
ax[1].plot([r['step'] for r in val], [r['accuracy']*100 for r in val], label='acc', marker='o')
ax[1].plot([r['step'] for r in val], [r['nontrivial_acc']*100 for r in val], label='nontrivial acc', marker='o')
ax[1].set_title('val accuracy (%)'); ax[1].set_xlabel('step'); ax[1].legend()
plt.tight_layout(); plt.show()